# Market rate and incentive

Which market rate, lag and incentive definition works best? Each version retrains the notebook 04 model with only the
incentive changed, and is scored on the validation years (2019–2021). Lower log loss is better.

In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import log_loss, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE = ROOT / "data" / "combined" / "model_table.parquet"  # built by notebook 03
PMMS = ROOT / "data" / "fred" / "MORTGAGE30US.csv"          # Freddie Mac PMMS 30-year rate, weekly
PMMS_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
sys.path.insert(0, str(ROOT / "source"))
from diagnostics import cpr, check, plot_over_time, shap_summary

# same features and settings as notebook 04, minus incentive (added per version)
FEATURES = ["age", "orig_upb", "fico", "ltv", "cltv", "dti", "num_borrowers", "num_units", "month",
            "first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel", "state", "agency"]
PARAMS = dict(objective="binary", learning_rate=0.1, num_leaves=63, min_data_in_leaf=2000,
              feature_fraction=0.8, bagging_fraction=0.5, bagging_freq=1, verbose=-1)

## 1. PMMS, monthly average

In [2]:
if not PMMS.exists():
    PMMS.parent.mkdir(parents=True, exist_ok=True)
    pl.read_csv(PMMS_URL).write_csv(PMMS)

pmms = (pl.read_csv(PMMS, try_parse_dates=True)
          .rename({"observation_date": "date", "MORTGAGE30US": "pmms"})
          .with_columns(t=(pl.col("date").dt.year() * 12 + pl.col("date").dt.month() - 1).cast(pl.Int32))  # same index as the table
          .group_by("t").agg(pl.col("pmms").mean()))

df = pl.read_parquet(TABLE)
for lag in [1, 2, 3]:  # rate from `lag` months before the payoff month
    df = df.join(pmms.select(t=pl.col("t") + lag, **{f"pmms_lag{lag}": "pmms"}), on="t", how="left")

# how far the notebook 03 rate (average rate on new loans in our sample) is from PMMS
df.select(gap=pl.col("mkt_rate") - pl.col("pmms_lag1")).describe()

statistic,gap
str,f64
"""count""",4.0037594e7
"""null_count""",0.0
"""mean""",0.098547
"""std""",0.308651
"""min""",-1.328678
"""25%""",-0.062447
"""50%""",0.141171
"""75%""",0.310065
"""max""",0.832501


## 2. Incentive definitions

- **diff**: loan rate − market rate (notebook 03)
- **ratio**: loan rate / market rate
- **savings**: monthly payment saved by refinancing the current balance over the remaining term, in dollars (bigger loans save more)

In [3]:
def payment(rate):
    r, n = rate / 1200, (361 - pl.col("age")).clip(1, None)  # monthly rate, months left
    return pl.col("exposure") * r / (1 - (1 + r).pow(-n))

def incentive(rate, how):
    return {"diff": pl.col("orig_rate") - rate,
            "ratio": pl.col("orig_rate") / rate,
            "savings": payment(pl.col("orig_rate")) - payment(rate)}[how]

## 3. Train and score each version

In [4]:
data = {s: df.filter(pl.col("split") == s) for s in ["train", "valid", "test"]}
X = {s: d.select(FEATURES).to_pandas() for s, d in data.items()}
y = {s: d["prepay"].to_numpy() for s, d in data.items()}

models = {}  # (rate, incentive) -> trained model, so the test section reuses them

def set_incentive(rate_col, how):
    for s in data:
        X[s]["incentive"] = data[s].select(incentive(pl.col(rate_col), how)).to_series().to_numpy()

def fit_score(rate_col, how, split="valid"):
    set_incentive(rate_col, how)
    if (rate_col, how) not in models:
        models[rate_col, how] = lgb.train(PARAMS, lgb.Dataset(X["train"], y["train"]), num_boost_round=1000,
                      valid_sets=[lgb.Dataset(X["valid"], y["valid"])], callbacks=[lgb.early_stopping(50, verbose=False)])
    model = models[rate_col, how]
    p, w = model.predict(X[split], num_iteration=model.best_iteration), data[split]["exposure"].to_numpy()
    return dict(rate=rate_col, incentive=how, rounds=model.best_iteration,
                log_loss=log_loss(y[split], p), auc=roc_auc_score(y[split], p),
                actual_cpr=cpr((y[split] * w).sum() / w.sum()), predicted_cpr=cpr((p * w).sum() / w.sum()))

**Market rate and lag** (incentive = diff). `mkt_rate` is the notebook 03 rate, lagged 1 month.

In [5]:
lags = pl.DataFrame([fit_score(r, "diff") for r in ["mkt_rate", "pmms_lag1", "pmms_lag2", "pmms_lag3"]])
best_rate = lags.sort("log_loss")["rate"][0]
lags

rate,incentive,rounds,log_loss,auc,actual_cpr,predicted_cpr
str,str,i64,f64,f64,f64,f64
"""mkt_rate""","""diff""",85,0.099572,0.698731,26.046336,20.37136
"""pmms_lag1""","""diff""",57,0.099493,0.695193,26.046336,21.355492
"""pmms_lag2""","""diff""",55,0.099564,0.695287,26.046336,20.893644
"""pmms_lag3""","""diff""",49,0.099833,0.691526,26.046336,20.213051


**Incentive definition**, using the best rate above.

In [6]:
defs = pl.DataFrame([fit_score(best_rate, how) for how in ["diff", "ratio", "savings"]])
best = defs.sort("log_loss").row(0, named=True)
defs

rate,incentive,rounds,log_loss,auc,actual_cpr,predicted_cpr
str,str,i64,f64,f64,f64,f64
"""pmms_lag1""","""diff""",57,0.099493,0.695193,26.046336,21.355492
"""pmms_lag1""","""ratio""",58,0.099624,0.688186,26.046336,21.608214
"""pmms_lag1""","""savings""",60,0.099534,0.697165,26.046336,20.778737


## 4. Test years: best version

Scored once on 2022 onward. The table compares it with notebook 04; the charts are the standard check from
`source/diagnostics.py` for the best version. The error columns are the average miss in CPR points across the buckets of
each chart (lower is better).

In [7]:
v = (best["rate"], best["incentive"])
display(pl.DataFrame([fit_score("mkt_rate", "diff", split="test"), fit_score(*v, split="test")]))  # notebook 04 vs. best

set_incentive(*v)
model = models[v]
errors, tables = check(data["test"].with_columns(incentive=incentive(pl.col(best["rate"]), "diff")),
                       {"best": model.predict(X["test"], num_iteration=model.best_iteration)})
errors

rate,incentive,rounds,log_loss,auc,actual_cpr,predicted_cpr
str,str,i64,f64,f64,f64,f64
"""mkt_rate""","""diff""",85,0.040626,0.683617,8.371516,11.459917
"""pmms_lag1""","""diff""",57,0.040144,0.684232,8.371516,9.594385


model,incentive_pp_error,age_years_error
str,f64,f64
"""best""",1.437168,1.226631


**Monthly CPR**, all years: actual vs. best version.

In [8]:
plot_over_time(pl.concat(data.values()),
               {"best": np.concatenate([model.predict(X[s], num_iteration=model.best_iteration) for s in data])})

**SHAP summary**, test years: each dot is a loan-month. Right of zero = the feature makes that loan faster; red = high value.

In [9]:
shap_summary(model, X["test"], title="SHAP, test years: best")

feature,shap_pct,direction
str,f64,str
"""incentive""",49.539156,"""higher → faster"""
"""age""",15.808285,"""higher → faster"""
"""state""",6.976482,""""""
"""orig_upb""",5.075626,"""higher → faster"""
"""first_time_buyer""",3.202681,""""""
…,…,…
"""agency""",1.309548,""""""
"""dti""",1.016255,"""higher → slower"""
"""channel""",0.8943,""""""


## 5. Takeaways

- **Use PMMS lagged 1 month.** It has the best validation log loss. Longer lags are worse, and 3 months is the worst.
- **Keep the simple incentive (diff).** Ratio and dollar savings don't beat it on log loss.
- **The validation differences are small** (log loss 0.09949 vs. 0.09957). Our sample's rate is close to PMMS on average (0.10 pp higher) but swings from −1.3 to +0.8 pp.
- **The test years improve clearly:** predicted CPR 9.6 vs. 11.5 before (actual 8.4), and the average miss halves on both charts (incentive 3.0 → 1.4 CPR, age 3.1 → 1.2 CPR).
- **The fix is just past zero incentive.** The old model jumped too early (5–7 CPR too fast between 0 and +1 pp); the new one tracks actual there.
- **Still wrong:** locked-in loans (incentive below −2 pp) are predicted at about 6–7 CPR vs. 2–4 actual, and the 2020–21 boom is still too slow.
- **SHAP:** incentive's share rises from 46% to 50%, so the model leans more on the cleaner rate.
- **Next:** switch notebook 03 to PMMS lagged 1 month and retrain notebook 04.